# 연평균 수익률

In [54]:
# 필요한 라이브러리 불러오기
import yfinance as yf
import pandas as pd
from datetime import datetime

# --- 1. 종목 설정 ---
# 분석할 종목들의 한글 이름과 yfinance에서 사용되는 티커(Symbol)를 매핑한 딕셔너리
tickers = {
    "TSLL": "TSLL",
    "테슬라": "TSLA",
    "TLT": "TLT",
    "CONY": "CONY",
    "애플": "AAPL",
    "TLTW": "TLTW",
    "AMD": "AMD",
    "VOO": "VOO",
    "QQQ": "QQQ",
    "월마트": "WMT",
    "비자": "V",
    "버크셔 해서웨이B": "BRK-B"
}

# --- 2. 데이터 다운로드 ---
# 오늘 날짜를 기준으로 yfinance에서 종가(Close) 데이터만 다운로드
end_date = datetime.today()
data = yf.download(list(tickers.values()), end=end_date)['Close']

# --- 3. 연평균 수익률 계산 ---
# 종목별로 연평균 수익률(CAGR, 복리 수익률)을 계산
results = {}
for label, symbol in tickers.items():
    try:
        # 만약 티커가 다운로드된 데이터에 없다면 처리
        if symbol not in data.columns:
            results[label] = "데이터 없음"
            continue

        # 종가 시계열 가져오기 및 결측치 제거
        series = data[symbol].dropna()

        # 데이터가 너무 적을 경우 예외 처리
        if len(series) < 2:
            results[label] = "데이터 부족"
            continue

        # 시작 가격, 종료 가격
        start_price = series.iloc[0]
        end_price = series.iloc[-1]

        # 데이터 시작일 기준으로 투자 기간(년 단위) 계산
        start_date = series.index[0]
        duration_years = (end_date - start_date).days / 365.25

        # 복리 수익률(CAGR) 공식 적용
        annual_return = (end_price / start_price) ** (1 / duration_years) - 1

        # 결과 저장: % 형식의 문자열로 저장
        results[label] = f"{annual_return * 100:.2f}%"

    except Exception as e:
        # 에러 발생 시 오류 메시지 저장
        results[label] = f"오류: {e}"

# --- 4. 결과 출력 ---
# 각 종목의 연평균 수익률을 출력
print("전체 기간 기준 연평균 수익률:")
for label, result in results.items():
    print(f"{label:<15}: {result}")

# --- 5. 숫자 리스트 생성 ---
# 문자열로 저장된 수익률에서 '%' 제거하고 float형으로 변환한 리스트 생성
result_list = [
    float(results[label].strip('%')) if isinstance(results[label], str) and results[label].endswith('%')
    else None
    for label in tickers.keys()
]

# 수익률을 퍼센트 단위에서 소수점 단위로 변환
result_list = [
    r / 100 if r is not None else None
    for r in result_list
]

# 수익률 리스트 출력
print("\n연평균 수익률 리스트:")
print(result_list)


[*********************100%***********************]  12 of 12 completed

전체 기간 기준 연평균 수익률:
TSLL           : -18.51%
테슬라            : 42.53%
TLT            : 3.74%
CONY           : 47.69%
애플             : 18.65%
TLTW           : -3.11%
AMD            : 8.35%
VOO            : 14.27%
QQQ            : 10.04%
월마트            : 18.61%
비자             : 21.74%
버크셔 해서웨이B      : 11.06%

연평균 수익률 리스트:
[-0.18510000000000001, 0.4253, 0.0374, 0.4769, 0.1865, -0.0311, 0.08349999999999999, 0.1427, 0.10039999999999999, 0.1861, 0.21739999999999998, 0.1106]


# 리스크 표준편차

In [66]:
import yfinance as yf
import pandas as pd
import numpy as np
from datetime import datetime

# --- 1. 분석 대상 종목 정의 ---
# 종목 이름 (한글 키)과 티커 (Yahoo Finance에서 사용하는 코드) 매핑
tickers = {
    "TSLL": "TSLL",
    "테슬라": "TSLA",
    "TLT": "TLT",
    "CONY": "CONY",
    "애플": "AAPL",
    "TLTW": "TLTW",
    "AMD": "AMD",
    "VOO": "VOO",
    "QQQ": "QQQ",
    "월마트": "WMT",
    "비자": "V",
    "버크셔 해서웨이B": "BRK-B"
}

# 오늘 날짜 기준으로 데이터 조회 종료일 설정
end_date = datetime.today()

# --- 2. 종가(Close) 데이터 다운로드 ---
# yfinance를 통해 각 종목의 전체 가용기간에 대해 종가를 가져옴
data = yf.download(list(tickers.values()), end=end_date)['Close']

# --- 3. 일간 수익률 계산 ---
# (오늘 가격 - 어제 가격) / 어제 가격
returns = data.pct_change().dropna()

# --- 4. 연간 변동성 계산 ---
# 일간 수익률의 표준편차 × √252 × 100 (% 단위)
# 252는 연간 거래일 수를 의미함 (평균적으로 미국 주식시장 기준)
volatility = returns.std() * np.sqrt(252) * 100

# --- 5. 종목별 연간 변동성 결과 정리 ---
# '종목', '티커', '연간 표준편차 (%)'로 구성된 데이터프레임 생성
result = pd.DataFrame({
    '종목': list(tickers.keys()),      # 한글 이름
    '티커': list(tickers.values()),    # 실제 티커
    '연간 표준편차 (%)': [
        f"{volatility.get(t, 'N/A'):.2f}" if t in volatility else "데이터 없음" 
        for t in tickers.values()
    ]
})

# --- 6. 결과 출력 ---
print("전체 기간 기준 연간 표준편차 (변동성):")
print(result)

# --- 7. 최적화 등에 활용 가능한 순수 숫자 리스트 생성 ---
# 수치 계산이 안 되는 종목은 NaN 처리됨
volatility_list = [volatility.get(t, np.nan) for t in tickers.values()]

print("\n변동성 리스트 (숫자 형태):")
print(volatility_list)


[*********************100%***********************]  12 of 12 completed

전체 기간 기준 연간 표준편차 (변동성):
           종목     티커 연간 표준편차 (%)
0        TSLL   TSLL      124.28
1         테슬라   TSLA       64.96
2         TLT    TLT       15.64
3        CONY   CONY       60.70
4          애플   AAPL       28.18
5        TLTW   TLTW       11.27
6         AMD    AMD       50.25
7         VOO    VOO       16.41
8         QQQ    QQQ       21.65
9         월마트    WMT       21.44
10         비자      V       18.64
11  버크셔 해서웨이B  BRK-B       16.62

변동성 리스트 (숫자 형태):
[np.float64(124.2792369433197), np.float64(64.95695647415593), np.float64(15.63501069644996), np.float64(60.70343585578456), np.float64(28.175146606117224), np.float64(11.267867167462327), np.float64(50.24625747508892), np.float64(16.411068144909194), np.float64(21.65148833845047), np.float64(21.44472464766378), np.float64(18.636814652420103), np.float64(16.62175843676644)]


## 선형 계획법

In [68]:
from pulp import LpMaximize, LpProblem, LpVariable, lpSum, value, LpStatus, LpStatusOptimal
import pandas as pd

# --- 1. 데이터 정의 ---
# 각 종목의 기대 수익률과 표준편차(연간 변동성)를 기반으로 데이터프레임 생성
data = {
    'Stock': ['TSLL', 'TSLA', 'TLT', 'CONY', 'AAPL', 'TLTW', 'AMD', 'VOO', 'QQQ', 'WMT', 'V', 'BRK-B'],
    'Expected_Return': result_list,      
    'Std_Dev': volatility_list           
}
df = pd.DataFrame(data)

# --- 2. 파생 데이터 계산 ---
risk_free_rate = 0.02  # 무위험 수익률 2%
# Sharpe 비율 계산: (기대 수익 - 무위험 수익) / 리스크
df['Sharpe'] = (df['Expected_Return'] - risk_free_rate) / df['Std_Dev']

# 상위 30% 종목만 별도로 표시 (우량 종목으로 간주)
sharpe_threshold = df['Sharpe'].quantile(0.70)
df['High_Sharpe'] = df['Sharpe'] >= sharpe_threshold

# 리스크 스코어 계산: 표준편차를 0~1로 정규화
df['Risk_Score'] = (df['Std_Dev'] - df['Std_Dev'].min()) / (df['Std_Dev'].max() - df['Std_Dev'].min())

# --- 3. 파라미터 설정 ---
total_budget = 1000  # 총 투자 예산 (만원 단위)
min_alloc_pct = 0.03  # 일반 종목 최소 투자 비율 (3%)
max_alloc_pct = 0.15  # 모든 종목 최대 투자 비율 (15%)
boosted_min_pct = 0.08  # 우량 종목(High Sharpe)에 대해 최소 8% 투자 유도

# 포트폴리오 전체 리스크 허용 한도: 평균 리스크 점수의 1.5배
max_portfolio_risk_score = df['Risk_Score'].mean() * total_budget * 1.5

# --- 4. 선형계획 모델 정의 ---
# 수익을 최대화하는 LP 모델 정의
model = LpProblem("Integrated_Sharpe_Risk_Optimization", LpMaximize)

# --- 5. 변수 정의 ---
# 각 종목별 투자 금액을 변수로 정의 (0 이상)
investments = LpVariable.dicts("Invest", df['Stock'], lowBound=0)

# --- 6. 목적 함수: 기대 수익 최대화 ---
# sum(Expected_Return × 투자금) 최대화
model += lpSum(df.loc[df['Stock'] == s, 'Expected_Return'].values[0] * investments[s] for s in df['Stock'])

# --- 7. 제약 조건 설정 ---

# (1) 총 투자금 제약: 총합이 정확히 1000만원이어야 함
model += lpSum(investments[s] for s in df['Stock']) == total_budget, "Total_Budget"

# (2) 전체 리스크 스코어 제한
model += lpSum(df.loc[df['Stock'] == s, 'Risk_Score'].values[0] * investments[s] for s in df['Stock']) <= max_portfolio_risk_score, "Max_Risk"

# (3) 종목별 최소/최대 비중 제약
for s in df['Stock']:
    row = df[df['Stock'] == s].iloc[0]
    # 우량 종목은 더 많은 최소 투자 비중
    min_pct = boosted_min_pct if row['High_Sharpe'] else min_alloc_pct
    model += investments[s] >= min_pct * total_budget, f"Min_{s}"
    model += investments[s] <= max_alloc_pct * total_budget, f"Max_{s}"

# --- 8. 최적화 실행 ---
model.solve()

# --- 9. 결과 출력 ---
print(f"\n최적화 상태: {LpStatus[model.status]}")
if model.status == LpStatusOptimal:
    print(f"최적 해를 찾았습니다!")
    print(f"총 기대 수익: {value(model.objective):.2f} 만원")
    print(f"리스크 허용 한도: {max_portfolio_risk_score:.2f}")

    print("\n투자 포트폴리오 구성:")
    results = []
    total_risk_score = 0
    for s in df['Stock']:
        amt = investments[s].varValue  # 투자 금액
        row = df[df['Stock'] == s].iloc[0]
        risk_contrib = row['Risk_Score'] * amt  # 종목의 리스크 기여도
        total_risk_score += risk_contrib
        results.append({
            'Stock': s,
            'Invest (만원)': f"{amt:.2f}",
            'Return': f"{row['Expected_Return']:.4f}",
            'Sharpe': f"{row['Sharpe']:.4f}",
            'High Sharpe?': row['High_Sharpe'],
            'Allocation (%)': f"{amt / total_budget * 100:.2f}%",
            'Risk Score': f"{risk_contrib:.2f}"
        })

    # 결과를 표 형태로 출력
    df_result = pd.DataFrame(results)
    print(df_result.to_string(index=False))
    print(f"\n총 포트폴리오 리스크 스코어: {total_risk_score:.2f}")
else:
    print("최적 해를 찾을 수 없습니다. 제약 조건을 완화하거나 데이터 확인이 필요합니다.")



최적화 상태: Optimal
최적 해를 찾았습니다!
총 기대 수익: 236.86 만원
리스크 허용 한도: 348.21

투자 포트폴리오 구성:
Stock Invest (만원)  Return  Sharpe  High Sharpe? Allocation (%) Risk Score
 TSLL       30.00 -0.1851 -0.0017         False          3.00%      30.00
 TSLA      150.00  0.4253  0.0062         False         15.00%      71.26
  TLT       30.00  0.0374  0.0011         False          3.00%       1.16
 CONY      150.00  0.4769  0.0075          True         15.00%      65.62
 AAPL      150.00  0.1865  0.0059         False         15.00%      22.44
 TLTW       30.00 -0.0311 -0.0045         False          3.00%       0.00
  AMD       30.00  0.0835  0.0013         False          3.00%      10.35
  VOO       80.00  0.1427  0.0075          True          8.00%       3.64
  QQQ       30.00  0.1004  0.0037         False          3.00%       2.76
  WMT      140.00  0.1861  0.0077          True         14.00%      12.61
    V      150.00  0.2174  0.0106          True         15.00%       9.78
BRK-B       30.00  0.1106  0.00